In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import os

SITE_INFO_FILE = "../SiteInfo.csv"
FORCING_DIR = "/stu01/dongwz/data/inputdata/single_point/urban_flux/v1"
MOD_DIR = "/tera10/yuanhua/dongwz/mksrf/srf_5x5"
ETH_DIR = "/tera12/yuanhua/data/CoLMrawdata/vegetation_morphology/tree_height_ETH"
GFCC_DIR = "/tera12/yuanhua/data/CoLMrawdata/urban_ecology/tree_fraction_GFCC"
OUTPUT_FILE = "HTOP.csv"

igbp = ['Ocean', 'NET', 'BET', 'NDT', 'BDT', 'MF', 'CS', 'OP', 'WS', 'SVA', 'Grass', 'WetLand', 'CropLand', 'Urban', 'CropMosaics', 'SnowandIce', 'Barren', 'Water']

info = pd.read_csv(SITE_INFO_FILE)

rows = []

for _, site in info.iterrows():
    site_name = site['site']
    target_lat = site['Lat']
    target_lon = site['Lon']

    if target_lat >= 0:
        reg_slat = int(target_lat / 5) * 5
        reg_elat = int(target_lat / 5) * 5 + 5
    else:
        reg_slat = int(target_lat / 5) * 5 - 5
        reg_elat = int(target_lat / 5) * 5

    if target_lon >= 0:
        reg_slon = int(target_lon / 5) * 5
        reg_elon = int(target_lon / 5) * 5 + 5
    else:
        reg_slon = int(target_lon / 5) * 5 - 5
        reg_elon = int(target_lon / 5) * 5

    print(reg_elat, reg_slat, reg_slon, reg_elon)

    region = f"RG_{reg_elat}_{reg_slon}_{reg_slat}_{reg_elon}"
    eth_file = os.path.join(ETH_DIR, f"{region}.Htop500m.ETH.nc")
    mod_file = os.path.join(MOD_DIR, f"{region}.MOD2020.nc")
    gfcc_file = os.path.join(GFCC_DIR, f"{region}.PCTT500m.GFCC.2015.nc")
    obs_file = os.path.join(FORCING_DIR, f"{site_name}_metforcing_v1.nc")

    print(eth_file)
    missing_files = [
        path for path in [eth_file, mod_file, gfcc_file, obs_file]
        if not os.path.exists(path)
    ]
    if missing_files:
        print(f"Skip {site_name}: missing {', '.join(missing_files)}")
        continue

    with (
        xr.open_dataset(obs_file) as obs_ds,
        xr.open_dataset(mod_file) as mod_ds,
        xr.open_dataset(eth_file) as eth_ds,
        xr.open_dataset(gfcc_file) as gfcc_ds,
    ):
        lat_idx = abs(mod_ds.lat - target_lat).argmin().item()
        lon_idx = abs(mod_ds.lon - target_lon).argmin().item()

        lc = mod_ds['LC'].isel(lat=lat_idx, lon=lon_idx).values
        slat = mod_ds['lat'].isel(lat=lat_idx).values
        slon = mod_ds['lon'].isel(lon=lon_idx).values

        print(f"{site_name} Location {slat} {slon}")
        print(f"Land Cover Type is {igbp[int(lc)]}")

        eth_htop = eth_ds['HTOP'].isel(lat=lat_idx, lon=lon_idx).values
        gfcc_tree = gfcc_ds['PCT_Tree'].isel(lat=lat_idx, lon=lon_idx).values / 100

        obs_htop = obs_ds['tree_mean_height'].values[0, 0]
        obs_tree = obs_ds['tree_area_fraction'].values[0, 0]

    rows.append({
        'Site': site_name,
        'ETH_htop': eth_htop,
        'Site_htop': obs_htop,
        'GFCC': gfcc_tree,
        'Site_Tree': obs_tree,
    })

if not rows:
    raise RuntimeError("No site data were extracted. HTOP.csv was not overwritten.")

df = pd.DataFrame(rows, columns=['Site', 'ETH_htop', 'Site_htop', 'GFCC', 'Site_Tree'])
df.to_csv(OUTPUT_FILE, index=False)


-35 -40 145 150
/tera12/yuanhua/data/CoLMrawdata/vegetation_morphology/tree_height_ETH/RG_-35_145_-40_150.Htop500m.ETH.nc
AU-Preston Location -37.73125076293945 145.01458740234375
Land Cover Type is Urban
-35 -40 145 150
/tera12/yuanhua/data/CoLMrawdata/vegetation_morphology/tree_height_ETH/RG_-35_145_-40_150.Htop500m.ETH.nc
AU-SurreyHills Location -37.827083587646484 145.09791564941406
Land Cover Type is Urban
50 45 -125 -120
/tera12/yuanhua/data/CoLMrawdata/vegetation_morphology/tree_height_ETH/RG_50_-125_45_-120.Htop500m.ETH.nc
CA-Sunset Location 49.22708511352539 -123.07707977294922
Land Cover Type is Urban
65 60 20 25
/tera12/yuanhua/data/CoLMrawdata/vegetation_morphology/tree_height_ETH/RG_65_20_60_25.Htop500m.ETH.nc
FI-Kumpula Location 60.202083587646484 24.960416793823242
Land Cover Type is Urban
65 60 20 25
/tera12/yuanhua/data/CoLMrawdata/vegetation_morphology/tree_height_ETH/RG_65_20_60_25.Htop500m.ETH.nc
FI-Torni Location 60.16875076293945 24.93958282470703
Land Cover Type 